# 03 — Advanced Model (Sentence-BERT + HistGradientBoosting)

This notebook implements the production-grade semantic matching model using Sentence-BERT embeddings and engineered pair features.

In [2]:

import os

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['USE_TF'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split

/Users/vansh/Desktop/ATS_Match/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load prepared dataset

In [3]:
data_path = Path('data/processed/resume_jd_clean.csv')
if not data_path.exists():
    raise FileNotFoundError('Processed dataset not found. Run notebook 01 first.')

df = pd.read_csv(data_path)
df.head(2)

,resume_text,jd_text,label,sample_id
0,summaryhighly motivated sales associate with e...,net2source inc. is an award-winning total work...,0,0
1,professional summarycurrently working with cat...,at salas obrien we tell our clients that were ...,0,1


## Generate Sentence-BERT embeddings

In [4]:
sbert_model_name = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(sbert_model_name)

resume_emb = embedder.encode(
    df['resume_text'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
jd_emb = embedder.encode(
    df['jd_text'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

resume_emb.shape, jd_emb.shape

Batches: 100%|██████████| 195/195 [00:29<00:00,  6.53it/s]


((6240, 384), (6240, 384))

## Feature engineering

Pairwise features include:
- Resume and JD dense embeddings
- Absolute difference
- Hadamard product
- Cosine similarity
- L2 distance

In [5]:
dot_product = np.sum(resume_emb * jd_emb, axis=1, keepdims=True)
cosine_sim = dot_product
l2_distance = np.linalg.norm(resume_emb - jd_emb, axis=1, keepdims=True)
abs_diff = np.abs(resume_emb - jd_emb)
hadamard = resume_emb * jd_emb

X = np.hstack([resume_emb, jd_emb, abs_diff, hadamard, cosine_sim, l2_distance]).astype(np.float32)
y = df['label'].to_numpy().astype(int)

X.shape, y.shape

((6240, 1538), (6240,))

## Train/Test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

((4992, 1538), (1248, 1538))

## Model tuning with GridSearchCV

In [7]:
base_model = HistGradientBoostingClassifier(
    loss='log_loss',
    random_state=42,
    class_weight='balanced',
)

param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [None, 8],
    'max_leaf_nodes': [31, 63],
    'min_samples_leaf': [20],   
    'l2_regularization': [0.0, 0.01],
    'max_iter': [200],
}

grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print('Best Params:', grid.best_params_)
print('Best CV F1:', round(grid.best_score_, 4))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


Exception ignored in: <function ResourceTracker.__del__ at 0x106858cc0>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x101ebccc0>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/m

Best Params: {'l2_regularization': 0.01, 'learning_rate': 0.05, 'max_depth': None, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
Best CV F1: 0.7397


## Evaluate on holdout set

In [8]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob),
}
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print('\nClassification report:')
print(classification_report(y_test, y_pred, digits=4))

accuracy: 0.7492
precision: 0.7380
recall: 0.7677
f1: 0.7526
roc_auc: 0.8348

Classification report:
              precision    recall  f1-score   support

           0     0.7612    0.7309    0.7457       628
           1     0.7380    0.7677    0.7526       620

    accuracy                         0.7492      1248
   macro avg     0.7496    0.7493    0.7492      1248
weighted avg     0.7497    0.7492    0.7491      1248



## Ranking metrics for one JD against multiple resumes\n
\n
For recruiter workflows, the model is used as a **ranking engine**, not only a pairwise classifier.\n
\n
- **Precision@K**: among the top-K ranked resumes, how many are truly relevant.\n
- **NDCG@K**: quality of ranking order with higher weight to relevant resumes appearing earlier in the list.\n
\n
In this offline setup, each query uses one positive Resume-JD pair as ground truth and several sampled negative resumes as distractors.

In [9]:
def build_features_for_query(jd_vector: np.ndarray, resume_vectors: np.ndarray) -> np.ndarray:
    jd_matrix = np.repeat(jd_vector.reshape(1, -1), repeats=resume_vectors.shape[0], axis=0).astype(np.float32)
    resume_matrix = resume_vectors.astype(np.float32)
    cosine_sim = np.sum(resume_matrix * jd_matrix, axis=1, keepdims=True)
    l2_distance = np.linalg.norm(resume_matrix - jd_matrix, axis=1, keepdims=True)
    abs_diff = np.abs(resume_matrix - jd_matrix)
    hadamard = resume_matrix * jd_matrix
    return np.hstack([resume_matrix, jd_matrix, abs_diff, hadamard, cosine_sim, l2_distance]).astype(np.float32)

def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    order = np.argsort(-y_score)[:k]
    return float(np.sum(y_true[order]) / k)

def ndcg_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    order = np.argsort(-y_score)[:k]
    gains = y_true[order]
    discounts = 1.0 / np.log2(np.arange(2, len(gains) + 2))
    dcg = float(np.sum(gains * discounts))

    ideal_gains = np.sort(y_true)[::-1][:k]
    ideal_discounts = 1.0 / np.log2(np.arange(2, len(ideal_gains) + 2))
    idcg = float(np.sum(ideal_gains * ideal_discounts))
    return 0.0 if idcg == 0.0 else dcg / idcg


idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
    
test_pos = idx_test[df.iloc[idx_test]['label'].to_numpy() == 1]
test_pool = np.array(idx_test)
rng = np.random.default_rng(42)

K = 5
NEG_PER_QUERY = 19  # total candidates per query = 20
max_queries = min(200, len(test_pos))

p_at_k_scores = []
ndcg_scores = []

for q_idx in test_pos[:max_queries]:
    candidate_pool = test_pool[test_pool != q_idx]
    sampled_negs = rng.choice(candidate_pool, size=min(NEG_PER_QUERY, len(candidate_pool)), replace=False)
    candidates = np.concatenate([[q_idx], sampled_negs])

    jd_vec = jd_emb[q_idx]
    resume_vecs = resume_emb[candidates]
    X_query = build_features_for_query(jd_vec, resume_vecs)
    scores = best_model.predict_proba(X_query)[:, 1]

    relevance = np.zeros(len(candidates), dtype=int)
    relevance[0] = 1

    p_at_k_scores.append(precision_at_k(relevance, scores, k=min(K, len(candidates))))
    ndcg_scores.append(ndcg_at_k(relevance, scores, k=min(K, len(candidates))))

print(f"Queries evaluated: {len(p_at_k_scores)}")
print(f"Precision@{K}: {np.mean(p_at_k_scores):.4f}")
print(f"NDCG@{K}: {np.mean(ndcg_scores):.4f}")

Queries evaluated: 200
Precision@5: 0.1210
NDCG@5: 0.3999


## Persist model artifact

In [ ]:
artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, artifacts_dir / 'hist_gbc_model.joblib')
print(f"Saved model to: {(artifacts_dir / 'hist_gbc_model.joblib').resolve()}")

Saved model to: /Users/vansh/Desktop/ATS_Match/artifacts/hist_gbc_model.joblib


Exception ignored in: <function ResourceTracker.__del__ at 0x1073d8cc0>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x105820cc0>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/m